# 02 — Modeling, Evaluation & Calibration

A thin narrative over `matchodds.modeling`: compare the four approaches — the bookmaker-implied
**baseline**, multinomial **logistic regression**, **XGBoost**, and **Dixon-Coles** — under
**strictly temporal** cross-validation on proper scoring rules (**log-loss** + **Brier**; accuracy
secondary), then inspect calibration. All logic lives in the `matchodds` package; the cells only
import and call it — the same code that `make train` freezes to `models/v1.joblib`.

> Run `make data && make features` first, or point `MATCHODDS_DATA_DIR` at the committed sample.

## The feature table

Built in-memory from the master matches table (leakage-free; see `01_data`).

In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from matchodds.features import pipeline
from matchodds.modeling import calibration, cv, metrics, train
from matchodds.modeling.xgboost_model import XGBoostModel

table = pipeline.build(write=False)
print(f"{len(table):,} matches | {table['date'].min()} -> {table['date'].max()}")
print("outcome balance H/D/A:", table["result"].value_counts(normalize=True).round(3).to_dict())
table.head()

## Temporal-CV bake-off

Every candidate is scored under one forward-chaining `TimeOrderedSplit` on the same table. The
discriminative models are evaluated in their **deployable, calibrated** form; the bookmaker baseline
is a strong benchmark but is **never shipped** (the serving request carries no odds), so it is scored
with `deployable=False`. Selection minimises mean CV **log-loss** among the deployable models.

In [ ]:
metadata = train.run(table, Path(tempfile.mkdtemp()))

comparison = pd.DataFrame(metadata["models"]).T[["log_loss", "brier", "accuracy", "deployable"]]
comparison = comparison.sort_values("log_loss")
print(f"selected: {metadata['selected_model']} | calibration: {metadata['calibration_method']}")
comparison

## Reliability: before vs. after calibration

Proper scoring rewards *honest* probabilities, not just the right winner. Below, XGBoost is fit on
all but the final temporal fold and scored on it — raw vs. **hold-out (prefit) sigmoid-calibrated** —
with each predicted class-probability binned against its observed frequency. A perfectly calibrated
model sits on the diagonal. Dixon-Coles is generative and naturally calibrated, so it is
reliability-*checked*, not wrapped.

**Calibration construction (Epic 06.5).** Inside `calibration.calibrate`, the base estimator is fit
on the inner training slice (the expanding-window earlier portion of `safe_calibration_cv`'s final
fold), then wrapped in `sklearn.frozen.FrozenEstimator` and handed to `CalibratedClassifierCV` to
fit the calibration regressor on the strictly-later held-out tail — leakage-free. The previous
`CalibratedClassifierCV(cv=temporal_split, ensemble=True)` averaged per-fold submodels and
sigmoid-squashed confident predictions enough to **invert** the real home advantage on near-even
fixtures (e.g. `AEK(H)` vs `PAOK`: 0.472 raw → 0.336 deployed). The hold-out scheme above preserves
the ordering (`AEK(H)` vs `PAOK` now reads 0.434).

In [ ]:
splitter = cv.TimeOrderedSplit(table["date"])
*_, (train_idx, test_idx) = splitter.split()
train_table, test_table = table.iloc[train_idx], table.iloc[test_idx]
y_test = metrics.encode_labels(test_table["result"])

uncalibrated = XGBoostModel().fit(train_table).predict_proba(test_table)
calibrated = calibration.calibrate(
    XGBoostModel(),
    train_table,
    method="sigmoid",
    cv=calibration.safe_calibration_cv(train_table),
).predict_proba(test_table)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
for ax, proba, title in zip(axes, (uncalibrated, calibrated), ("Uncalibrated", "Calibrated (sigmoid)")):
    mean_predicted, observed, counts = metrics.reliability_curve(y_test, proba)
    drawn = counts > 0
    ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="perfect")
    ax.plot(mean_predicted[drawn], observed[drawn], "o-", label="XGBoost")
    ax.set(title=title, xlabel="Mean predicted probability")
    ax.legend(loc="upper left")
axes[0].set_ylabel("Observed frequency")
fig.suptitle(f"XGBoost reliability on the final temporal fold ({len(test_table)} matches)")
fig.tight_layout()

## Takeaways

- Selection is on **log-loss**, never accuracy; the baseline is scored as a benchmark but excluded
  from selection (it needs closing odds the service never receives).
- `make train` runs exactly this bake-off on the full data and freezes the winning **calibrated**
  model to `models/v1.joblib` with a `v1.metadata.json` sidecar (per-model CV scores, selected model,
  calibration method, feature columns, seed, data version, library versions).
- Calibration tightens the reliability curve toward the diagonal without re-ranking outcomes —
  honest probabilities are the headline deliverable here.
- Epic 06.5 swapped the per-fold ensemble calibration for the hold-out (prefit) scheme above,
  restoring the home-advantage ordering on near-even fixtures and improving the shipped logistic
  model's mean CV log-loss (0.9989 → 0.9945) and Brier (0.5954 → 0.5930).